<a href="https://colab.research.google.com/github/Bernpro/Just-do-it/blob/main/Neural_Network_SMS_Text_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Configuración
max_len = 50
trunc_type = 'post'
padding_type = 'post'
oov_tok = "<OOV>"

# Tokenizar el texto
tokenizer = Tokenizer(num_words=1000, oov_token=oov_tok)
tokenizer.fit_on_texts(train_df['message'])

# Convertir a secuencias y añadir padding
train_sequences = tokenizer.texts_to_sequences(train_df['message'])
train_padded = pad_sequences(train_sequences, maxlen=max_len, padding=padding_type, truncating=trunc_type)

test_sequences = tokenizer.texts_to_sequences(test_df['message'])
test_padded = pad_sequences(test_sequences, maxlen=max_len, padding=padding_type, truncating=trunc_type)

# Convertir etiquetas (ham=0, spam=1)
train_labels = train_df['label'].map({'ham': 0, 'spam': 1}).values
test_labels = test_df['label'].map({'ham': 0, 'spam': 1}).values



model = tf.keras.Sequential([
    tf.keras.layers.Embedding(1000, 16, input_length=max_len),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(24, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid') # Sigmoid para 0 (ham) o 1 (spam)
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(train_padded, train_labels, epochs=30, validation_data=(test_padded, test_labels), verbose=2)


def predict_message(pred_msg):
    # Procesar el mensaje de entrada igual que el entrenamiento
    seq = tokenizer.texts_to_sequences([pred_msg])
    padded = pad_sequences(seq, maxlen=max_len, padding=padding_type, truncating=trunc_type)

    # Predecir
    prediction = model.predict(padded)[0][0]

    label = "spam" if prediction >= 0.5 else "ham"

    return [float(prediction), label]


